In [69]:
import sys
sys.path.insert(0, '/Users/miranda/code/mit/summer26/ahri/SIRL/miranda/src')
from utils import *
from tversky_sirl_2 import *

FPE_SPLIT_SEED = 42
SIRL_SEED = 0
n_queries = 1000

print(f"\n=== TRAINING SIRL FOR {n_queries} QUERIES ===")
anchors, positives, negatives = load_data(n_queries)
print(anchors.shape)

print(f"  -- seed {SIRL_SEED} --")
set_all_seeds(SIRL_SEED)

model, _ = train_tversky_sirl(
    anchors, positives, negatives,
    fbank_size=4,
    use_symmetric_loss=True,
)


=== TRAINING SIRL FOR 1000 QUERIES ===
(1000, 21, 27)
  -- seed 0 --
Epoch    0 | loss=4267075840.0000 | triplet_acc=0.463 | lr=0.00400
Epoch  100 | loss=75327760.0000 | triplet_acc=0.462 | lr=0.00400
Epoch  200 | loss=28519340.0000 | triplet_acc=0.462 | lr=0.00399
Epoch  300 | loss=10756356.0000 | triplet_acc=0.463 | lr=0.00399
Epoch  400 | loss=5427636.0000 | triplet_acc=0.463 | lr=0.00398
Epoch  500 | loss=3423467.7500 | triplet_acc=0.463 | lr=0.00398
Epoch  600 | loss=2032623.5000 | triplet_acc=0.463 | lr=0.00398
Epoch  700 | loss=1214578.6250 | triplet_acc=0.465 | lr=0.00397
Epoch  800 | loss=732416.7500 | triplet_acc=0.464 | lr=0.00397
Epoch  900 | loss=626961.5625 | triplet_acc=0.464 | lr=0.00396
Epoch 1000 | loss=359995.1875 | triplet_acc=0.464 | lr=0.00396
Epoch 1100 | loss=221214.5312 | triplet_acc=0.467 | lr=0.00396
Epoch 1200 | loss=177173.2812 | triplet_acc=0.467 | lr=0.00395
Epoch 1300 | loss=122563.9922 | triplet_acc=0.467 | lr=0.00395
Epoch 1400 | loss=77094.0156 | tri

In [70]:
all_trajs = load_all_trajs()
all_features = get_features(all_trajs, all_trajs)

all_trajs shape: (10000, 21, 97)
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis elem

In [71]:
normalized_all_trajs = tx(all_trajs)
all_embeds = model(torch.tensor(normalized_all_trajs)).detach()
all_embeds.shape # (N, D)

torch.Size([10000, 6])

In [72]:
normalized_all_trajs.shape

(10000, 21, 27)

In [73]:
model

TverskySIRL(
  (encoder): Sequential(
    (0): TverskyProjection(
      (feature_bank): Embedding(4, 567)
      (prototypes): Embedding(6, 567)
    )
  )
  (tversky_sim): TverskySimilarity(
    (feature_bank): Embedding(4, 6)
  )
)

In [74]:
# get feature bank of the TverskyProjection layer in the encoder
feature_bank = model.encoder[0].feature_bank.weight.detach()
feature_bank.shape # 4 Tversky features, of 567-dim input vectors

torch.Size([4, 567])

In [75]:
# from tversky/test_mnist.py
# edited
import torch.nn.functional as F
def compute_salience(x, feature_bank):
    """
    x is (N, D)
    feature_bank is (F, D)
    """
    feature_measures = x @ feature_bank.T # (N, F)
    salience_measures = F.relu(feature_measures).sum(-1)
    return salience_measures

In [76]:
all_salience = compute_salience(torch.tensor(normalized_all_trajs).flatten(start_dim=1), feature_bank)

In [77]:
import sys
sys.path.insert(0, '..')
import numpy as np
import plotly.graph_objects as go
from src.envs.jacorobot import Jacorobot
from src.utils.bullet_utils import waypts_to_xyz

def vis_trajectory(traj, title=""):
    env_params = {
        "object_centers": {'HUMAN_CENTER': [-0.2, -0.6, 0.9], 'LAPTOP_CENTER': [-0.5, 0.0, 0.635]},
        "resources_dir": "../data/resources",
        "horizon": 10,
        "timestep": 0.5,
        "real": False,
    }

    env = Jacorobot(
        env_params["object_centers"],
        env_params["resources_dir"],
        env_params["horizon"],
        env_params["timestep"],
        debug=False  # headless — no GUI needed
    )

    xyz = waypts_to_xyz(env.objectID["robot"], traj)

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='lines+markers'
    ))

    posH = env_params["object_centers"]["HUMAN_CENTER"]
    posL = env_params["object_centers"]["LAPTOP_CENTER"]

    fig.add_trace(go.Scatter3d(
        x=[posH[0], posL[0]],
        y=[posH[1], posL[1]],
        z=[posH[2], posL[2]],
        mode='markers',
        marker=dict(
            color=["gray", "black"],
            symbol=["cross", "square"],
            size=8,
            showscale=False
        ),
        name="human (+), laptop (square)"
    ))
    fig.update_layout(title=title)

    config = {
    'toImageButtonOptions': {
        'format': 'png', # or jpeg, svg, webp
        'scale': 4 # Increase this for higher resolution
    }
    }
    fig.show(config=config)


In [78]:

min_salience_traj = all_trajs[np.argmin(all_salience).item()]
max_salience_traj = all_trajs[np.argmax(all_salience).item()]
print(max(all_salience))

tensor(151.5864)


In [79]:
vis_trajectory(max_salience_traj, "max salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [80]:
vis_trajectory(min_salience_traj, "min salience traj")

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis chest
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis neck
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_ankle
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis right_shoulder
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0,0) axis left_hip
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
urdfdom: no axis element for Joint, defaulting to (1,0

In [81]:
tversky_feature_measures = torch.tensor(normalized_all_trajs).flatten(start_dim=1) @ feature_bank.T
tversky_feature_measures.shape

torch.Size([10000, 4])

In [82]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# to numpy
tv = tversky_feature_measures.detach().cpu().numpy()   # (10000, 4)
gt = all_features

gt_names = ["table", "laptop"]
tv_names = [f"tversky_{k}" for k in range(tv.shape[1])]

# rows = ground truth (2), cols = tversky feature (4)
fig = make_subplots(
    rows=len(gt_names), cols=tv.shape[1],
    shared_xaxes=False, shared_yaxes=False,
    horizontal_spacing=0.05, vertical_spacing=0.12,
    subplot_titles=[
        f"{g} vs {t} (r={np.corrcoef(gt[:, gi], tv[:, ti])[0,1]:.2f})"
        for gi, g in enumerate(gt_names) for ti, t in enumerate(tv_names)
    ],
)

for gi in range(len(gt_names)):
    for ti in range(tv.shape[1]):
        fig.add_trace(
            go.Scattergl(
                x=gt[:, gi], y=tv[:, ti],
                mode="markers",
                marker=dict(size=3, opacity=0.35),
                showlegend=False,
            ),
            row=gi + 1, col=ti + 1,
        )
        fig.update_xaxes(title_text=gt_names[gi], row=gi + 1, col=ti + 1)
        fig.update_yaxes(title_text=tv_names[ti], row=gi + 1, col=ti + 1)

fig.update_layout(height=600, width=1300, title="Tversky feature measures vs ground-truth features")
config = {
  'toImageButtonOptions': {
    'format': 'png', # or jpeg, svg, webp
    'scale': 4 # Increase this for higher resolution
  }
}
fig.show(config=config)

# semantic algebra?

In [83]:
# from tversky-networks-iclr2026/experiments/103-vision-nabirds/src/semantic_utils.py
def retrieve_semantic_expression(
    instance_vectors: torch.Tensor,   # (N, D)  — all instance reps from parquet
    feature_bank: torch.Tensor,        # (F, D)  — from .npy
    expression: str,
    top_feature_count: int,
    top_result_count: int,
) -> dict:
    """
    Evaluate a set expression over instance vectors and feature bank.
    Expression uses s(i) notation where i is a dataset item_id (row index).
    Example: "s(0) - s(1)"  →  features of item 0 minus features of item 1
    """
    query_item_ixes = []

    def s(item_ix: int) -> set:
        print(item_ix)
        feature_values = instance_vectors[item_ix:item_ix+1] @ feature_bank.T  # (1, F)
        print("feature_values")
        print(feature_values)
        feature_ixes = []
        for feature_ix in torch.argsort(feature_values[0], descending=True)[:top_feature_count]:
            print("loop")
            print(feature_values[0][feature_ix])
            if feature_values[0][feature_ix] > 0:
                feature_ixes.append(int(feature_ix))
            else:
                break
        query_item_ixes.append(item_ix)
        print(set(feature_ixes))
        return set(feature_ixes)

    semantic_features = eval(expression)

    if not semantic_features:
        # logging.warning("Expression evaluated to empty feature set.")
        return {"expression": expression, "query_item_ixes": [], "top_instances": [], "feature_count": 0}

    semantic_f_bank = torch.index_select(
        feature_bank, 0, torch.tensor(sorted(semantic_features))
    )
    dot = instance_vectors @ semantic_f_bank.T          # (N, |features|)
    p_saliences = F.relu(dot).sum(dim=1)                # (N,)
    p_measures  = dot.sum(dim=1)                        # (N,)

    top_instances = []
    for result_ix in torch.argsort(p_measures, descending=True)[:top_result_count]:
        top_instances.append({
            "item_ix"  : result_ix.item(),
            "salience" : p_saliences[result_ix].item(),
            "measure"  : p_measures[result_ix].item(),
        })

    return {
        "expression"      : expression.strip(),
        "query_item_ixes" : query_item_ixes,
        "feature_count"   : len(semantic_features),
        "top_instances"   : top_instances,
    }

In [84]:
table_min_traj = all_trajs[np.argmin(all_features[:,0])]
table_max_traj = all_trajs[np.argmax(all_features[:,0])]
laptop_min_traj = all_trajs[np.argmin(all_features[:,1])]
laptop_max_traj = all_trajs[np.argmax(all_features[:,1])]

In [85]:
normalized_trajs = tx(np.array([
    table_min_traj,
    table_max_traj,
    laptop_min_traj,
    laptop_max_traj
]))
normalized_trajs.shape

(4, 21, 27)

In [86]:
retrieve_semantic_expression(
    instance_vectors=torch.tensor(normalized_trajs).flatten(start_dim=1), #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression="s(3)-s(2)", # table_max - table_min
    top_feature_count=4,
    top_result_count=10,
)

3
feature_values
tensor([[10.8924, 12.5709, 11.6097, 12.1812]])
loop
tensor(12.5709)
loop
tensor(12.1812)
loop
tensor(11.6097)
loop
tensor(10.8924)
{0, 1, 2, 3}
2
feature_values
tensor([[27.7771, 30.8872, 31.1948, 32.0806]])
loop
tensor(32.0806)
loop
tensor(31.1948)
loop
tensor(30.8872)
loop
tensor(27.7771)
{0, 1, 2, 3}


{'expression': 's(3)-s(2)',
 'query_item_ixes': [],
 'top_instances': [],
 'feature_count': 0}

# centering?

In [87]:
all_norm_trajs = torch.tensor(normalized_all_trajs).flatten(start_dim=1)
mu = all_norm_trajs.mean(0)        # shape [6]
centered_norm_trajs = all_norm_trajs - mu     # [10000, 6]
centered_all_salience = compute_salience(all_norm_trajs, feature_bank).detach()

In [88]:
examples = torch.tensor(normalized_trajs).flatten(start_dim=1)
centered_examples = examples - mu

In [89]:
laptop_min_idx = np.argmin(all_features[:,1])
laptop_max_idx = np.argmax(all_features[:,1])

min_max_res = retrieve_semantic_expression(
    instance_vectors=centered_norm_trajs, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression=f"s({laptop_min_idx})-s({laptop_max_idx})",
    top_feature_count=4,
    top_result_count=10,
)
max_min_res = retrieve_semantic_expression(
    instance_vectors=centered_norm_trajs, #(N, D)
    feature_bank=feature_bank, # (F, D)
    expression=f"s({laptop_max_idx})-s({laptop_min_idx})",
    top_feature_count=4,
    top_result_count=10,
)

4894
feature_values
tensor([[15.4364, 18.3752, 18.6676, 18.8611]])
loop
tensor(18.8611)
loop
tensor(18.6676)
loop
tensor(18.3752)
loop
tensor(15.4364)
{0, 1, 2, 3}
5018
feature_values
tensor([[-1.4483,  0.0589, -0.9175, -1.0383]])
loop
tensor(0.0589)
loop
tensor(-0.9175)
{1}
5018
feature_values
tensor([[-1.4483,  0.0589, -0.9175, -1.0383]])
loop
tensor(0.0589)
loop
tensor(-0.9175)
{1}
4894
feature_values
tensor([[15.4364, 18.3752, 18.6676, 18.8611]])
loop
tensor(18.8611)
loop
tensor(18.6676)
loop
tensor(18.3752)
loop
tensor(15.4364)
{0, 1, 2, 3}


In [90]:
max_min_laptop_vals = []
for instance in max_min_res["top_instances"]:
    idx = instance["item_ix"]
    max_min_laptop_vals.append(all_features[idx, 1])
print(np.mean(max_min_laptop_vals))

min_max_laptop_vals= []
for instance in min_max_res["top_instances"]:
    idx = instance["item_ix"]
    min_max_laptop_vals.append(all_features[idx, 1])
print(np.mean(min_max_laptop_vals))

nan
0.24060081317557716
